## AI模型推理实战：从PyTorch到ONNX到Tensor RT
在深度学习模型的整个生命周期中，训练只是上半场。当你的模型在GPU上以令人满意的准确率完成训练后，真正的挑战才刚刚开始——如何让它在生产环境中以最低延迟、最高吞吐量地运行？

这个问题在过去几年里变得越来越紧迫。大模型的浪潮推动着算力需求的指数级增长，而硬件资源的增长速度却远远跟不上。在这种背景下，模型推理优化已经从"锦上添花"变成了"刚需"。

本篇文章将带你走进AI模型推理优化的实战世界。我们选择经典的ResNet50作为实验对象，因为它既是计算机视觉领域的里程碑式模型，又具有适中的复杂度，非常适合作为推理优化的"试金石"。我们将通过三种主流推理框架的横向对比，完整展示推理的过程，并对3种推理框架做个简要对比：

- **PyTorch原生推理** —— 作为性能基线和参照

- **ONNX Runtime** —— 跨平台、高性能的推理引擎，通过图优化和算子融合来加速

- **TensorRT** —— NVIDIA官方出品的推理SDK，利用内核自动调优和低精度量化将性能推向极致

环境搭建：环境需要英伟达的GPU。
python库为：torch,onnx,onnxtime,tensorrt
官方文档是了解这些库的最佳文档：[ONNX](https://onnx.ai/onnx/intro/), [ONNX Runtime](https://onnxruntime.ai/docs/tutorials/),   [tensrRT](https://docs.nvidia.com/deeplearning/tensorrt/latest/architecture/architecture-overview.html)

### 模型获取
在模型获取阶段，我们通常会先准备好已经训练完成的模型权重文件。在 PyTorch 中，常见的保存方式有两种：

- 保存 `state_dict`：这是最推荐的方式，文件通常以 `.pth` 或 `.pt` 结尾，保存的是模型参数张量的字典结构。恢复模型时，需要先定义模型结构，然后用 `load_state_dict()` 将权重加载进去。可参考[PyTorch深度学习入门]({{< ref "/post/ai-inference-practice" >}})
- 保存整个模型：可以直接保存 `torch.save(model, path)`，但这种方式会把模型结构和参数一起序列化，跨环境、跨版本加载时更容易出现兼容性问题。

在推理优化场景下，PyTorch 的 `.pth` / `.pt` 权重文件虽然适合训练与继续训练，但并不是最理想的部署格式。这里就引出 ONNX 格式的优势：

- **跨框架可移植**：ONNX 是一个开放的模型交换格式，可以在多个推理引擎之间共享，如 ONNX Runtime、TensorRT、OpenVINO 等。
- **推理性能优化**：ONNX 模型是一个静态计算图，许多推理引擎可以对这个图进行算子融合、常量折叠、内核自动调优等性能优化。
- **硬件适配更灵活**：借助 ONNX，可以更方便地把模型部署到不同硬件上，包括 CPU、GPU、NPU、FPGA 等。

因此，在模型获取之后，通常会把训练好的 PyTorch 模型导出为 ONNX 格式，然后再进入下一步的 ONNX Runtime / TensorRT 推理优化流程。

本文将使用训练好的resnet50模型，ResNet-50 是一个在计算机视觉领域非常经典的深度学习模型，可以通俗地理解为一个功能强大的“视觉理解”核心。

它的核心功能是图像分类，能够识别出图片里的主要物体是什么。它可以直接从互联网上获取。

In [ ]:
import urllib.request
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
onnx_model_url = "https://s3.amazonaws.com/onnx-model-zoo/resnet/resnet50v2/resnet50v2.tar.gz"
imagenet_labels_url = "https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json"

urllib.request.urlretrieve(onnx_model_url, filename="resnet50v2.tar.gz")
urllib.request.urlretrieve(imagenet_labels_url, filename="imagenet-simple-labels.json")

### 模型推理

#### 图片预处理

在将图片输入模型之前，我们需要统一图像尺寸和数值范围，否则不同尺寸、不同像素分布的图片可能导致推理结果不稳定。

本节中，`convert_image_size(path)` 和 `preprocess(input_data)` 这两个函数正是完成这一预处理流程的关键：

- `convert_image_size(path)`：打开图片并检查尺寸。如果图片不是目标大小 224×224，就使用高质量的 Lanczos 重采样进行缩放。这样可以保证进入模型的每张图片都有一致的空间分辨率。
- `preprocess(input_data)`：将图片数据转换为模型所需的张量形式。这个函数会把颜色通道数据转换为 `float32`，先归一化到 `[0,1]`，再减去 ImageNet 预训练模型对应的均值并除以标准差。最后把数据 reshape 成模型接受的 `(1, 3, 224, 224)` 形状。

这种预处理方式对应于 ResNet50 预训练模型在训练时的输入规范，能够最大程度减少训练与推理之间的分布差异，从而提高预测结果的准确性和稳定性。

下面的代码会先读取一张图片并统一尺寸，然后将其转换为浮点数组、做标准化处理，最终生成可以直接送入模型进行推理的输入数据。

In [ ]:
def convert_image_size(path):
    image = Image.open(path)
    print("raw Image size: ", image.size)

    if image.size != (224, 224):
        resized_img = image.resize((224,224),Image.Resampling.LANCZOS)
        return resized_img
    return image

def preprocess(input_data):
    img_data = input_data.astype('float32')

    mean_vec = np.array([0.485, 0.456, 0.406])
    stddev_vec = np.array([0.229, 0.224, 0.225])

    norm_img_data = np.zeros(img_data.shape).astype('float32')
    for i in range(img_data.shape[0]):
        norm_img_data[i,:,:] = (img_data[i,:,:]/255 - mean_vec[i]) / stddev_vec[i]
    norm_img_data = norm_img_data.reshape(1,3,224,224).astype('float32')

    return norm_img_data

#### PyTorch原生推理

本节展示的是 PyTorch 原生推理的核心流程：

1. 读取图片，统一尺寸与格式；
2. 进行输入标准化，得到符合模型预训练规范的张量；
3. 加载 ResNet50 模型并切换到 `eval()` 模式；
4. 使用 `torch.inference_mode()` 进行前向推理；
5. 计算推理时间，并返回最可能的分类结果。

这段代码的重点在于“本地推理”的实现方式。它使用 torchvision 提供的预训练 ResNet50，在 GPU 或 CPU 上运行，并通过 `time.time()` 统计推理耗时。

在生产环境中，PyTorch 原生推理也可以被包装成一个远程推理服务。远程推理的核心逻辑与本地推理相同，区别在于：

- 客户端先完成 `convert_image_size()` 和 `preprocess()` 这类预处理；
- 经过网络发送至远程服务器，例如通过 HTTP/GRPC 请求；
- 服务器端接收输入后，执行本节代码中的 `model(input_data)` 推理；
- 最终结果再由服务器返回给客户端。

因此，`infer_by_torch()` 函数既可以作为本地调试和基准测试工具，也可以视为远程推理服务中最核心的推理单元。

In [ ]:
import torch
import torchvision
import numpy as np
import time
def infer_by_torch(path, device):
    labels = load_labels('imagenet-simple-labels.json')
    image = convert_image_size(path)
    image_data = np.array(image).transpose(2,0,1)
    input_data =  torch.as_tensor(preprocess(image_data)).to(device)
    print(input_data.shape)

    model = torchvision.models.resnet50(torchvision.models.ResNet50_Weights.DEFAULT).to(device)
    model.eval()

    start = time.time()
    with torch.inference_mode():
        output = model(input_data)
    idx = torch.argmax(output)
    end = time.time()
    infer_time = np.round((end -start) * 1000, 2)

    print('torch prediction is:'+ labels[idx])
    print('Inference time: '+ str(infer_time) + " ms")

#### onnx runtime推理

ONNX Runtime 的推理流程核心在于：

1. 使用 `convert_image_size()` 和 `preprocess()` 统一输入图像尺寸与数值范围；
2. 创建 `onnxruntime.InferenceSession` 加载已导出的 ONNX 模型；
3. 通过 `session.get_inputs()[0].name` 获取模型输入名；
4. 调用 `session.run()` 执行前向推理；
5. 对输出结果进行后处理，得到最终分类预测和推理耗时。

这种方式依赖静态计算图，因此 ONNX Runtime 能利用图优化和算子融合提高推理效率，并且更易于跨平台部署。

In [ ]:
import onnxruntime
import matplotlib.pyplot as plt
def infer_by_onnxruntime(path):
    labels = load_labels('imagenet-simple-labels.json')
    image = convert_image_size(path)

    print("Image size: ", image.size)
    
    plt.imshow(image)
    plt.axis('off')
    #plt.show()

    image_data = np.array(image).transpose(2,0,1)
    input_data = preprocess(image_data)

    session = onnxruntime.InferenceSession('model/resnet50v2.onnx', None)
    input_name = session.get_inputs()[0].name

    start = time.time()
    raw_result = session.run([], {input_name: input_data})
    end = time.time()
    res = postprocess(raw_result)

    infer_time = np.round((end -start) * 1000, 2)
    idx = np.argmax(res)

    print('final top prediction is:'+ labels[idx])
    print('Inference time: '+ str(infer_time) + " ms")

### tensor rt推理


In [ ]:
def convert_onnx_to_engine(path):
    import tensorrt as trt
    TRT_LOGGER = trt.Logger(trt.Logger.WARNING)
    builder = trt.Builder(TRT_LOGGER)
    network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
    parser = trt.OnnxParser(network, TRT_LOGGER)

    with open(path, 'rb') as model:
        if not parser.parse(model.read()):
            print('ERROR: Failed to parse the ONNX file.')
            for error in range(parser.num_errors):
                print(parser.get_error(error))
            return None
    config = builder.create_builder_config()
    config.set_flag(trt.BuilderFlag.FP16)
    config.max_workspace_size = 1 << 30

    engine = builder.build_cuda_engine(network, config)
    if engine is None:
        print("engine build error")
    return engine

def infer_by_tensorrt(path, device):
    import tensorrt as trt
    import pucuda.driver as cuda
    import pucuda.autoinit
    labels = load_labels('imagenet-simple-labels.json')
    image = convert_image_size(path)
    image_data = np.array(image).transpose(2,0,1)
    input_data =  torch.as_tensor(preprocess(image_data)).to(device)

    engine = convert_onnx_to_engine(path)
    context = engine.create_execution_context()

    input_name = "input"
    output_name = "output"
    input_shape = (1, 3, 224, 224)

    d_input = cuda.mem_alloc(np.prod(input_shape) * np.float32().itemsize)
    d_output = cuda.mem_alloc(1000 * np.float32().itemsize)

    context.set_tensor_address(input_name, int(d_input))
    context.set_tensor_address(output_name, int(d_output))

    context.execute_async_v3(stream_handle=0)

    output_data = np.empty(1000, dtype=np.float32)
    cuda.memcpy_dtoh(output_data, d_output)

    idx = np.argmax(output_data)

    print('torsorrt prediction is:'+ labels[idx])
    

### 三种推理方式的对比

从原理上看：

- **PyTorch 原生推理**：直接使用 PyTorch 模型进行前向计算，适合调试与开发。模型以动态图方式执行，灵活但优化空间有限。
- **ONNX Runtime**：先将 PyTorch 模型导出为 ONNX 静态图，再通过 ONNX Runtime 加载执行。它以静态计算图为基础，可做图优化、算子融合，适合跨平台部署。
- **TensorRT**：以 ONNX 为输入，构建 NVIDIA 专用的高效 CUDA 引擎，进一步做内核自动调优和低精度计算。它是面向 NVIDIA GPU 的极致加速方案。

结合本文代码：

- `infer_by_torch()` 直接加载 torchvision 的 ResNet50 模型并用 `torch.inference_mode()` 运行，最直接，代码实现简单。
- `infer_by_onnxruntime()` 先读取 ONNX 模型文件，再用 `onnxruntime.InferenceSession` 创建会话，调用 `session.run()` 完成推理，适合在不同平台上复用同一个模型文件。
- `infer_by_tensorrt()` 先将 ONNX 模型解析为 TensorRT 引擎，再分配 GPU 内存、设置输入输出地址，最后调用 `context.execute_async_v3()` 执行，适合追求最低延迟的 NVIDIA GPU 部署。

| 推理方式 | 支持平台 | 优点 | 缺点 | 适用场景 |
| --- | --- | --- | --- | --- |
| PyTorch 原生推理 |  |  |  |  |
| ONNX Runtime |  |  |  |  |
| TensorRT |  |  |  |  |